# Generador de Nube de Palabras

**Aplicación de escritorio con `tkinter` que lee múltiples formatos de archivo y genera nubes de palabras interactivas.**

---

## Formatos soportados

| Formato | Extensión | Biblioteca usada |
|---------|-----------|------------------|
| Texto plano | `.txt`, `.md`, `.log`, `.py`, `.xml` | `builtins` |
| PDF | `.pdf` | `pdfplumber` / `PyPDF2` |
| Word | `.docx` | `python-docx` |
| JSON | `.json` | `json` |
| CSV | `.csv` | `csv` |
| HTML | `.html`, `.htm` | `beautifulsoup4` |

---

## Tecnologías

- **GUI**: `tkinter` + `ttk` (estándar en Python)
- **Nube de palabras**: `wordcloud`
- **Visualización**: `matplotlib` embebido en `tkinter`
- **Imágenes**: `Pillow`
- **Stopwords**: `nltk`

---

# Heurísticas de Jakob Nielsen

Las **10 heurísticas de usabilidad de Jakob Nielsen** son principios generales para el diseño de interfaces de usuario. A continuación se detalla cómo se aplica cada una en esta aplicación:

---

### H1 · Visibilidad del estado del sistema
> *El sistema siempre debe mantener informados a los usuarios de lo que está ocurriendo, a través de retroalimentación apropiada dentro de un tiempo razonable.*


### H2 · Concordancia entre el sistema y el mundo real
> *El sistema debe hablar el idioma del usuario, con palabras, frases y conceptos familiares.*


### H3 · Control y libertad del usuario
> *Los usuarios a menudo eligen funciones del sistema por error y necesitan una "salida de emergencia" claramente marcada.*


### H4 · Consistencia y estándares
> *Los usuarios no deberían tener que preguntarse si diferentes palabras, situaciones o acciones significan lo mismo.*


### H5 · Prevención de errores
> *Mejor que un buen mensaje de error es un diseño cuidadoso que prevenga que ocurra el problema.*


### H6 · Reconocimiento en lugar de recuerdo
> *Minimizar la carga de memoria del usuario haciendo visibles los objetos, acciones y opciones.*

### H7 · Flexibilidad y eficiencia de uso
> *Los aceleradores —no vistos por el usuario novato— pueden acelerar la interacción para el usuario experto.*

### H8 · Diseño estético y minimalista
> *Los diálogos no deben contener información irrelevante o raramente necesaria.*

### H9 · Ayudar a los usuarios a reconocer, diagnosticar y recuperarse de errores
> *Los mensajes de error deben expresarse en lenguaje llano, indicar el problema con precisión y sugerir una solución.*

### H10 · Ayuda y documentación
> *Aunque es mejor que el sistema no necesite explicación adicional, puede ser necesario proporcionar ayuda.*
---

---
## Instalación de dependencias

In [27]:
# Instalar todas las dependencias necesarias
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'wordcloud', 'pillow', 'PyPDF2', 'python-docx',
    'matplotlib', 'nltk', 'pdfplumber', 'beautifulsoup4',
    '-q'
], check=True)
print(' Dependencias instaladas correctamente')

 Dependencias instaladas correctamente



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


---
## Imports y configuración inicial

In [28]:
#  Librería estándar 
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, colorchooser
import threading
import json
import re
import csv
import io
from pathlib import Path

#  Visualización 
from wordcloud import WordCloud, STOPWORDS
from PIL import Image, ImageTk
import matplotlib
matplotlib.use('TkAgg')           # backend tkinter
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

#  Procesamiento de lenguaje 
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

#  Lectura de archivos (opcionales, con detección de disponibilidad) 
try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except ImportError:
    HAS_PDFPLUMBER = False

try:
    import PyPDF2
    HAS_PYPDF2 = True
except ImportError:
    HAS_PYPDF2 = False

try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

try:
    from bs4 import BeautifulSoup
    HAS_BS4 = True
except ImportError:
    HAS_BS4 = False

print(f'pdfplumber : {HAS_PDFPLUMBER}')
print(f'PyPDF2     : {HAS_PYPDF2}')
print(f'python-docx: {HAS_DOCX}')
print(f'bs4        : {HAS_BS4}')
print(' Imports completados')

pdfplumber : True
PyPDF2     : True
python-docx: True
bs4        : True
 Imports completados


---
## Módulo de lectura de archivos

Cada función lee un formato específico y devuelve el texto como `str`. Todas implementan **fallback de codificación** para manejar archivos con distintas codificaciones.

El diccionario `FILE_READERS` mapea extensiones a funciones, permitiendo agregar nuevos formatos sin modificar la lógica de despacho. Esto sigue el **Principio Abierto/Cerrado**.

> **H5 (Prevención de errores):** El validador de extensiones rechaza archivos no soportados con un mensaje descriptivo antes de intentar la lectura.

In [29]:
def read_txt(path: str) -> str:
    """Lee archivo de texto con fallback de codificación."""
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            with open(path, 'r', encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo decodificar el archivo: {path}")


def read_pdf(path: str) -> str:
    """Lee PDF usando pdfplumber (preferido) o PyPDF2 como respaldo."""
    if HAS_PDFPLUMBER:
        with pdfplumber.open(path) as pdf:
            return '\n'.join(page.extract_text() or '' for page in pdf.pages)
    if HAS_PYPDF2:
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            return '\n'.join(page.extract_text() or '' for page in reader.pages)
    raise ImportError("Instala pdfplumber o PyPDF2 para leer archivos PDF.")


def read_docx(path: str) -> str:
    """Extrae texto de documento Word (.docx)."""
    if not HAS_DOCX:
        raise ImportError("Instala python-docx para leer archivos DOCX.")
    doc = DocxDocument(path)
    return '\n'.join(p.text for p in doc.paragraphs)


def read_json(path: str) -> str:
    """Extrae todos los valores de cadena de un JSON (recursivo)."""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = []
    def _extract(obj):
        if isinstance(obj, str):
            texts.append(obj)
        elif isinstance(obj, dict):
            for v in obj.values():
                _extract(v)
        elif isinstance(obj, list):
            for item in obj:
                _extract(item)
    _extract(data)
    return ' '.join(texts)


def read_csv(path: str) -> str:
    """Lee CSV y concatena todos los campos de texto."""
    texts = []
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            with open(path, 'r', encoding=enc, newline='') as f:
                for row in csv.reader(f):
                    texts.extend(row)
            break
        except UnicodeDecodeError:
            continue
    return ' '.join(texts)


def read_html(path: str) -> str:
    """Extrae texto de HTML eliminando etiquetas."""
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    if HAS_BS4:
        soup = BeautifulSoup(content, 'html.parser')
        return soup.get_text(separator=' ')
    return re.sub(r'<[^>]+>', ' ', content)


# Tabla de despacho: extensión  función lectora
FILE_READERS = {
    '.txt':  read_txt,
    '.md':   read_txt,
    '.py':   read_txt,
    '.xml':  read_txt,
    '.log':  read_txt,
    '.pdf':  read_pdf,
    '.docx': read_docx,
    '.json': read_json,
    '.csv':  read_csv,
    '.html': read_html,
    '.htm':  read_html,
}

SUPPORTED_EXTENSIONS = list(FILE_READERS.keys())


def read_file(path: str) -> str:
    """Punto de entrada unificado: lee el archivo según su extensión."""
    ext = Path(path).suffix.lower()
    if ext not in FILE_READERS:
        supported = ', '.join(SUPPORTED_EXTENSIONS)
        raise ValueError(
            f"Formato no soportado: '{ext}'\n"
            f"Formatos aceptados: {supported}"
        )
    return FILE_READERS[ext](path)


print(' Módulo de lectura de archivos cargado')
print('  Formatos:', ', '.join(SUPPORTED_EXTENSIONS))

 Módulo de lectura de archivos cargado
  Formatos: .txt, .md, .py, .xml, .log, .pdf, .docx, .json, .csv, .html, .htm


---
## Módulo generador de nube de palabras

La función `generate_wordcloud` recibe el texto en bruto y todos los parámetros de configuración, y devuelve un objeto `WordCloud` listo para renderizar.

**Procesamiento del texto:**
1. Se combina el conjunto de stopwords de `wordcloud` con las de `nltk` para el idioma seleccionado.
2. Se eliminan caracteres no alfabéticos (números, puntuación) con una expresión regular.
3. Se aplica la expresión `regexp` para filtrar palabras de al menos 3 caracteres sin dígitos.

| Parámetro | Descripción |
|-----------|-------------|
| `max_words` | Número máximo de palabras (10–500) |
| `bg_color` | Color de fondo (nombre CSS o hex) |
| `colormap` | Paleta de matplotlib para colorear palabras |
| `lang` | Idioma para stopwords de nltk |
| `custom_stopwords` | Palabras adicionales a excluir |
| `width`, `height` | Dimensiones de la imagen generada |

In [30]:
# Mapas de opciones disponibles en la UI
STOPWORDS_LANGS = {
    'Español':   'spanish',
    'Inglés':    'english',
    'Francés':   'french',
    'Alemán':    'german',
    'Italiano':  'italian',
    'Portugués': 'portuguese',
    'Ninguno':   None,
}

COLOR_SCHEMES = {
    'Viridis':  'viridis',
    'Plasma':   'plasma',
    'Inferno':  'inferno',
    'Magma':    'magma',
    'Coolwarm': 'coolwarm',
    'Blues':    'Blues',
    'Reds':     'Reds',
    'Greens':   'Greens',
}


def _build_stopwords(lang_key: str, custom: set) -> set:
    """Combina stopwords de wordcloud + nltk + palabras personalizadas."""
    sw = set(STOPWORDS)
    lang = STOPWORDS_LANGS.get(lang_key)
    if lang:
        try:
            sw |= set(stopwords.words(lang))
        except OSError:
            pass
    sw |= custom
    return sw


def generate_wordcloud(
    text: str,
    max_words: int = 100,
    bg_color: str = 'white',
    colormap: str = 'viridis',
    lang: str = 'Español',
    custom_stopwords: set = None,
    width: int = 800,
    height: int = 500,
) -> WordCloud:
    """Genera y devuelve un objeto WordCloud a partir del texto dado."""
    if not text or not text.strip():
        raise ValueError("El texto está vacío. No se puede generar la nube.")

    sw = _build_stopwords(lang, custom_stopwords or set())

    # Conservar solo letras (incluye tildes y ñ)
    clean_text = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', text)

    wc = WordCloud(
        width=width,
        height=height,
        background_color=bg_color,
        max_words=max_words,
        stopwords=sw,
        colormap=colormap,
        collocations=True,
        min_font_size=10,
        max_font_size=100,
        prefer_horizontal=0.9,
        regexp=r'\b[^\d\W]{3,}\b',  # palabras de 3 letras sin dígitos
    ).generate(clean_text)

    return wc


print('Módulo generador de nube cargado')

Módulo generador de nube cargado


---
## Widget Tooltip

> **H6 (Reconocimiento en lugar de recuerdo)** · **H10 (Ayuda y documentación)**

La clase `Tooltip` añade una ventana emergente flotante a cualquier widget de tkinter cuando el cursor lo superpone. Cada control de la UI tiene su tooltip con descripción y atajo de teclado cuando aplica.

**Comportamiento:**
- Se muestra al entrar al widget (`<Enter>`)
- Desaparece al salir (`<Leave>`)
- Se posiciona automáticamente debajo del widget
- Fondo amarillo claro estándar de sistemas operativos

In [31]:
class Tooltip:
    """Tooltip flotante para cualquier widget tkinter. (H6, H10)"""

    def __init__(self, widget: tk.Widget, text: str):
        self.widget = widget
        self.text = text
        self._win = None
        widget.bind('<Enter>', self._show)
        widget.bind('<Leave>', self._hide)

    def _show(self, event=None):
        if self._win:
            return
        x = self.widget.winfo_rootx() + 20
        y = self.widget.winfo_rooty() + self.widget.winfo_height() + 4
        self._win = tw = tk.Toplevel(self.widget)
        tw.wm_overrideredirect(True)
        tw.wm_geometry(f'+{x}+{y}')
        tk.Label(
            tw, text=self.text, justify='left',
            background='#FFFFE0', relief='solid', borderwidth=1,
            font=('Segoe UI', 9), wraplength=240, padx=5, pady=3
        ).pack()

    def _hide(self, event=None):
        if self._win:
            self._win.destroy()
            self._win = None


print(' Clase Tooltip cargada')

 Clase Tooltip cargada


---
## Aplicación principal

Hereda de `tk.Tk` (ventana raíz). La UI se divide en tres zonas:

```

  BARRA SUPERIOR  [Título]         [Atajos: Ctrl+O G S Z]      H4, H7

                                                            
  PANEL IZQ.          ÁREA DE VISUALIZACIÓN                
  (280 px)        (Matplotlib embebido en tkinter)            H8
                                                            
  ARCHIVO        [Placeholder con instrucciones               H10
  OPCIONES        hasta que se genera la nube]             
  ACCIONES                                                  
                                                            

  BARRA DE ESTADO  [/// Mensaje]    [N palabras]          H1, H9

```

### Flujo de datos

```
Usuario abre archivo
     Hilo: read_file()         [H5: sin bloqueo de UI]
     self._current_text = text [H1: actualiza estado]

Usuario hace clic en Generar
     Guarda nube actual en _history  [H3: para undo]
     Hilo: generate_wordcloud()       [H5: sin bloqueo]
     Renderiza en matplotlib canvas   [H1: actualiza estado]

Usuario hace Ctrl+Z
     _history.pop()  re-renderiza    [H3: libertad]
```

In [32]:
class WordCloudApp(tk.Tk):
    """Aplicación de escritorio para generar nubes de palabras."""

    #  Paleta de colores centralizada (H4 - Consistencia) 
    THEME = {
        'bg':          '#F5F7FA',
        'panel':       '#FFFFFF',
        'accent':      '#4A90D9',
        'accent_dark': '#2D6EAA',
        'text':        '#2C3E50',
        'subtext':     '#7F8C8D',
        'border':      '#E0E4E8',
        'success':     '#27AE60',
        'error':       '#E74C3C',
        'warning':     '#F39C12',
    }

    def __init__(self):
        super().__init__()
        self.title('Nube de Palabras')
        self.geometry('1200x750')
        self.minsize(900, 600)
        self.configure(bg=self.THEME['bg'])

        # Estado interno
        self._current_text: str = ''
        self._current_file: str = ''
        self._wordcloud: WordCloud | None = None
        self._processing: bool = False
        self._history: list[WordCloud] = []  # historial para undo (H3)

        self._setup_style()
        self._build_ui()

        # Atajos de teclado (H7 - Flexibilidad y eficiencia)
        self.bind_all('<Control-o>', lambda e: self.open_file())
        self.bind_all('<Control-g>', lambda e: self.generate())
        self.bind_all('<Control-s>', lambda e: self.save_image())
        self.bind_all('<Control-z>', lambda e: self.undo())
        self.bind_all('<F5>',        lambda e: self.generate())
        self.bind_all('<Escape>',    lambda e: self._on_escape())

        # H1: mensaje de bienvenida que indica el primer paso
        self._set_status('Bienvenido — Abre un archivo para comenzar  (Ctrl+O)', 'info')
        self.protocol('WM_DELETE_WINDOW', self._on_close)

    #  Estilos ttk 

    def _setup_style(self):
        style = ttk.Style(self)
        style.theme_use('clam')
        T = self.THEME

        style.configure('TFrame',          background=T['bg'])
        style.configure('Panel.TFrame',    background=T['panel'])
        style.configure('TLabel',          background=T['panel'], foreground=T['text'],
                        font=('Segoe UI', 10))
        style.configure('Sub.TLabel',      background=T['panel'], foreground=T['subtext'],
                        font=('Segoe UI', 8))
        style.configure('Accent.TButton',  background=T['accent'], foreground='white',
                        font=('Segoe UI', 10, 'bold'), padding=(12, 6))
        style.map('Accent.TButton',
                  background=[('active', T['accent_dark']), ('disabled', '#B0BEC5')])
        style.configure('TButton',         font=('Segoe UI', 9), padding=(8, 4))
        style.configure('TCombobox',       font=('Segoe UI', 9))
        style.configure('TSpinbox',        font=('Segoe UI', 9))
        style.configure('TProgressbar',    troughcolor=T['border'],
                        background=T['accent'], thickness=6)

    #  Construcción de la interfaz 

    def _build_ui(self):
        self._build_top_bar()
        content = tk.Frame(self, bg=self.THEME['bg'])
        content.pack(fill='both', expand=True)
        self._build_left_panel(content)
        self._build_right_panel(content)
        self._build_status_bar()

    def _build_top_bar(self):
        """Barra superior con título y atajos de teclado. (H4, H7)"""
        T = self.THEME
        bar = tk.Frame(self, bg=T['accent'], height=52)
        bar.pack(fill='x')
        bar.pack_propagate(False)

        tk.Label(bar, text='Nube de Palabras', bg=T['accent'], fg='white',
                 font=('Segoe UI', 15, 'bold')).pack(side='left', padx=16)

        # Atajos visibles para usuarios novatos (H6)
        tk.Label(bar,
                 text='Ctrl+O Abrir · Ctrl+G Generar · Ctrl+S Guardar · Ctrl+Z Deshacer',
                 bg=T['accent'], fg='#D6EAF8',
                 font=('Segoe UI', 8)).pack(side='right', padx=16)

    def _build_left_panel(self, parent):
        """Panel izquierdo: archivo, opciones, botones de acción."""
        T = self.THEME

        left = tk.Frame(parent, bg=T['panel'], width=280,
                        highlightbackground=T['border'], highlightthickness=1)
        left.pack(side='left', fill='y')
        left.pack_propagate(False)

        inner = tk.Frame(left, bg=T['panel'])
        inner.pack(fill='both', expand=True, padx=16, pady=16)

        #  Sección ARCHIVO 
        ttk.Label(inner, text='ARCHIVO', style='Sub.TLabel').pack(anchor='w')

        self._file_var = tk.StringVar(value='Ningún archivo seleccionado')
        file_box = tk.Frame(inner, bg='#F8F9FA',
                            highlightbackground=T['border'], highlightthickness=1)
        file_box.pack(fill='x', pady=(4, 8))
        tk.Label(file_box, textvariable=self._file_var, bg='#F8F9FA',
                 fg=T['subtext'], font=('Segoe UI', 8), wraplength=220,
                 justify='left', pady=6, padx=6).pack(fill='x')

        open_btn = ttk.Button(inner, text='  Abrir archivo…', command=self.open_file)
        open_btn.pack(fill='x', pady=(0, 4))
        Tooltip(open_btn,
                'Abrir archivo (Ctrl+O)\n'
                'Formatos: TXT, PDF, DOCX, JSON, CSV, HTML, MD, XML, LOG, PY')

        ttk.Separator(inner, orient='horizontal').pack(fill='x', pady=10)

        #  Sección OPCIONES 
        ttk.Label(inner, text='OPCIONES', style='Sub.TLabel').pack(anchor='w')

        # Máx. palabras
        r1 = tk.Frame(inner, bg=T['panel'])
        r1.pack(fill='x', pady=3)
        ttk.Label(r1, text='Máx. palabras').pack(side='left')
        self._max_words = tk.IntVar(value=100)
        sp = ttk.Spinbox(r1, from_=10, to=500, increment=10,
                         textvariable=self._max_words, width=6)
        sp.pack(side='right')
        Tooltip(sp, 'Número máximo de palabras en la nube (10 – 500)')

        # Idioma
        r2 = tk.Frame(inner, bg=T['panel'])
        r2.pack(fill='x', pady=3)
        ttk.Label(r2, text='Idioma').pack(side='left')
        self._lang = tk.StringVar(value='Español')
        cb_lang = ttk.Combobox(r2, textvariable=self._lang, width=10,
                               values=list(STOPWORDS_LANGS.keys()), state='readonly')
        cb_lang.pack(side='right')
        Tooltip(cb_lang, 'Idioma para filtrar palabras vacías (artículos, preposiciones, etc.)')

        # Esquema de color
        r3 = tk.Frame(inner, bg=T['panel'])
        r3.pack(fill='x', pady=3)
        ttk.Label(r3, text='Esquema de color').pack(side='left')
        self._colormap = tk.StringVar(value='Viridis')
        cb_cm = ttk.Combobox(r3, textvariable=self._colormap, width=10,
                             values=list(COLOR_SCHEMES.keys()), state='readonly')
        cb_cm.pack(side='right')
        Tooltip(cb_cm, 'Paleta de colores para las palabras de la nube')

        # Color de fondo
        r4 = tk.Frame(inner, bg=T['panel'])
        r4.pack(fill='x', pady=3)
        ttk.Label(r4, text='Fondo').pack(side='left')
        self._bg_color = tk.StringVar(value='white')
        self._bg_swatch = tk.Label(r4, bg='white', width=4, height=1,
                                   relief='solid', bd=1, cursor='hand2')
        self._bg_swatch.pack(side='right')
        self._bg_swatch.bind('<Button-1>', self._pick_bg_color)
        Tooltip(self._bg_swatch, 'Clic para elegir el color de fondo de la nube')

        ttk.Separator(inner, orient='horizontal').pack(fill='x', pady=10)

        #  Sección EXCLUIR PALABRAS 
        ttk.Label(inner, text='EXCLUIR PALABRAS', style='Sub.TLabel').pack(anchor='w')
        ttk.Label(inner, text='(separadas por coma)', style='Sub.TLabel').pack(anchor='w')
        self._custom_sw = tk.Text(inner, height=3, font=('Segoe UI', 9),
                                  relief='solid', bd=1, wrap='word',
                                  bg='#FAFAFA', fg=T['text'])
        self._custom_sw.pack(fill='x', pady=(4, 8))
        Tooltip(self._custom_sw,
                'Palabras adicionales a excluir, separadas por coma.\n'
                'Ej: empresa, año, dato, valor')

        ttk.Separator(inner, orient='horizontal').pack(fill='x', pady=4)

        #  Botones de acción 
        gen_btn = ttk.Button(inner, text='  Generar nube  (F5)',
                             command=self.generate, style='Accent.TButton')
        gen_btn.pack(fill='x', pady=(8, 4))
        Tooltip(gen_btn, 'Generar la nube de palabras (Ctrl+G o F5)')

        save_btn = ttk.Button(inner, text='  Guardar imagen…', command=self.save_image)
        save_btn.pack(fill='x', pady=2)
        Tooltip(save_btn, 'Exportar la nube como PNG, JPG, SVG o PDF (Ctrl+S)')

        undo_btn = ttk.Button(inner, text='  Deshacer  (Ctrl+Z)', command=self.undo)
        undo_btn.pack(fill='x', pady=2)
        Tooltip(undo_btn,
                'Restaurar la nube de palabras anterior.\n'
                'Guarda hasta 10 estados de historial. (Ctrl+Z)')

        # Barra de progreso (H1 - Visibilidad del estado)
        self._progress = ttk.Progressbar(inner, mode='indeterminate')
        self._progress.pack(fill='x', pady=(10, 0))
        self._progress_lbl = ttk.Label(inner, text='', style='Sub.TLabel')
        self._progress_lbl.pack(anchor='w')

    def _build_right_panel(self, parent):
        """Área de visualización: placeholder y canvas matplotlib."""
        T = self.THEME

        right = tk.Frame(parent, bg=T['bg'])
        right.pack(side='left', fill='both', expand=True)

        self._canvas_frame = tk.Frame(right, bg=T['bg'])
        self._canvas_frame.pack(fill='both', expand=True, padx=16, pady=16)

        # Placeholder con instrucciones (H10 - Ayuda contextual)
        self._placeholder = tk.Frame(
            self._canvas_frame, bg='white',
            highlightbackground=T['border'], highlightthickness=1
        )
        self._placeholder.pack(fill='both', expand=True)

        ph = tk.Frame(self._placeholder, bg='white')
        ph.place(relx=0.5, rely=0.5, anchor='center')
        tk.Label(ph, text='', bg='white', fg=T['border'],
                 font=('Segoe UI', 72)).pack()
        tk.Label(ph, text='Abre un archivo y haz clic en Generar',
                 bg='white', fg=T['subtext'],
                 font=('Segoe UI', 13)).pack()
        tk.Label(ph,
                 text='Formatos: TXT · PDF · DOCX · JSON · CSV · HTML · MD · XML · LOG · PY',
                 bg='white', fg='#B0BEC5', font=('Segoe UI', 9)).pack(pady=4)

        # Figure matplotlib (oculto hasta la primera generación)
        self._fig, self._ax = plt.subplots(figsize=(9, 5.5))
        self._fig.patch.set_facecolor(T['bg'])
        self._ax.axis('off')
        self._mpl_canvas = FigureCanvasTkAgg(self._fig, master=self._canvas_frame)
        self._mpl_widget = self._mpl_canvas.get_tk_widget()

    def _build_status_bar(self):
        """Barra inferior de estado. (H1, H9)"""
        T = self.THEME
        bar = tk.Frame(self, bg='#ECF0F1',
                       highlightbackground=T['border'], highlightthickness=1)
        bar.pack(fill='x', side='bottom')

        self._status_icon = tk.StringVar(value='')
        self._status_var  = tk.StringVar(value='')

        tk.Label(bar, textvariable=self._status_icon, bg='#ECF0F1',
                 font=('Segoe UI', 10), width=2).pack(side='left', padx=(6, 0), pady=3)
        tk.Label(bar, textvariable=self._status_var,  bg='#ECF0F1',
                 fg=T['text'], font=('Segoe UI', 9), anchor='w').pack(side='left')

        self._word_count_var = tk.StringVar(value='')
        tk.Label(bar, textvariable=self._word_count_var, bg='#ECF0F1',
                 fg=T['subtext'], font=('Segoe UI', 8)).pack(side='right', padx=8)

    #  Manejadores de eventos 

    def open_file(self):
        """Abre diálogo de archivo y carga el contenido en un hilo. (H3, H5)"""
        filetypes = [
            ('Todos los soportados',
             '*.txt *.pdf *.docx *.json *.csv *.html *.htm *.md *.xml *.log *.py'),
            ('Texto plano',  '*.txt *.md *.log *.py *.xml'),
            ('PDF',          '*.pdf'),
            ('Word',         '*.docx'),
            ('JSON',         '*.json'),
            ('CSV',          '*.csv'),
            ('HTML',         '*.html *.htm'),
            ('Todos',        '*.*'),
        ]
        path = filedialog.askopenfilename(
            title='Seleccionar archivo',
            filetypes=filetypes,
        )
        if not path:
            return  # cancelado — sin consecuencias (H3)
        self._load_file_async(path)

    def _load_file_async(self, path: str):
        self._set_status(f'Leyendo {Path(path).name}…', 'info')
        self._start_progress('Leyendo archivo…')

        def _worker():
            try:
                text = read_file(path)
                if not text.strip():
                    self.after(0, lambda: self._on_load_error(
                        'El archivo está vacío o no contiene texto legible.'))
                    return
                self.after(0, lambda: self._on_file_loaded(path, text))
            except Exception as exc:
                msg = str(exc)
                self.after(0, lambda: self._on_load_error(msg))

        threading.Thread(target=_worker, daemon=True).start()

    def _on_file_loaded(self, path: str, text: str):
        self._stop_progress()
        self._current_file = path
        self._current_text = text
        name  = Path(path).name
        words = len(text.split())
        display = name if len(name) <= 35 else '…' + name[-32:]
        self._file_var.set(display)
        self._word_count_var.set(f'{words:,} palabras en el documento')
        self._set_status(f'\'{name}\' cargado — {words:,} palabras. Haz clic en Generar.', 'success')

    def _on_load_error(self, msg: str):
        self._stop_progress()
        # H9: mensaje de error claro con sugerencia de solución
        messagebox.showerror(
            'Error al leer archivo',
            f'{msg}\n\nVerifica que el archivo no esté dañado o protegido con contraseña.',
            parent=self
        )
        self._set_status('Error al leer el archivo. Intenta con otro.', 'error')

    def generate(self):
        """Inicia la generación de la nube en un hilo de fondo. (H1, H3, H5)"""
        if self._processing:
            return
        if not self._current_text:
            # H9: error amigable con instrucción de recuperación
            messagebox.showwarning(
                'Sin archivo',
                'Primero abre un archivo con Ctrl+O o el botón «Abrir archivo».',
                parent=self
            )
            return

        # Guardar estado actual para undo (H3 - Control del usuario)
        if self._wordcloud is not None:
            self._history.append(self._wordcloud)
            if len(self._history) > 10:
                self._history.pop(0)

        self._processing = True
        self._start_progress('Generando nube de palabras…')
        self._set_status('Procesando texto…', 'info')

        # Leer opciones de la UI
        max_words = self._max_words.get()
        lang      = self._lang.get()
        colormap  = COLOR_SCHEMES.get(self._colormap.get(), 'viridis')
        bg_color  = self._bg_color.get()
        raw_sw    = self._custom_sw.get('1.0', 'end').strip()
        custom_sw = {w.strip().lower() for w in raw_sw.split(',') if w.strip()}
        text      = self._current_text
        w = max(self._mpl_widget.winfo_width(),  600)
        h = max(self._mpl_widget.winfo_height(), 400)

        def _worker():
            try:
                wc = generate_wordcloud(
                    text, max_words=max_words, bg_color=bg_color,
                    colormap=colormap, lang=lang, custom_stopwords=custom_sw,
                    width=w, height=h,
                )
                self.after(0, lambda: self._on_generated(wc))
            except Exception as exc:
                msg = str(exc)
                self.after(0, lambda: self._on_generate_error(msg))

        threading.Thread(target=_worker, daemon=True).start()

    def _on_generated(self, wc: WordCloud):
        self._wordcloud  = wc
        self._processing = False
        self._stop_progress()

        # Mostrar canvas, ocultar placeholder
        self._placeholder.pack_forget()
        self._mpl_widget.pack(fill='both', expand=True)

        self._ax.clear()
        self._ax.imshow(wc, interpolation='bilinear')
        self._ax.axis('off')
        self._fig.tight_layout(pad=0)
        self._mpl_canvas.draw()

        n = len(wc.words_)
        self._set_status(f'Nube generada con {n} palabras únicas. Guarda con Ctrl+S.', 'success')
        self._word_count_var.set(f'{n} palabras en la nube')

    def _on_generate_error(self, msg: str):
        self._processing = False
        self._stop_progress()
        messagebox.showerror('Error al generar nube', msg, parent=self)
        self._set_status('Error al generar la nube. Revisa el archivo y las opciones.', 'error')

    def save_image(self):
        """Exporta la nube actual como imagen. (H7)"""
        if self._wordcloud is None:
            messagebox.showwarning('Sin nube', 'Genera una nube de palabras primero.', parent=self)
            return
        path = filedialog.asksaveasfilename(
            title='Guardar imagen',
            defaultextension='.png',
            filetypes=[
                ('PNG',  '*.png'),
                ('JPEG', '*.jpg'),
                ('SVG',  '*.svg'),
                ('PDF',  '*.pdf'),
            ],
            initialfile='nube_palabras',
        )
        if not path:
            return
        try:
            if Path(path).suffix.lower() in ('.svg', '.pdf'):
                self._fig.savefig(path, bbox_inches='tight', dpi=150)
            else:
                self._wordcloud.to_file(path)
            self._set_status(f'Imagen guardada: {Path(path).name}', 'success')
        except Exception as exc:
            messagebox.showerror('Error al guardar', str(exc), parent=self)

    def undo(self):
        """Restaura la nube de palabras anterior. (H3 - Control y libertad)"""
        if not self._history:
            self._set_status('No hay nube anterior para deshacer.', 'info')
            return
        wc = self._history.pop()
        self._wordcloud = wc
        self._ax.clear()
        self._ax.imshow(wc, interpolation='bilinear')
        self._ax.axis('off')
        self._fig.tight_layout(pad=0)
        self._mpl_canvas.draw()
        self._set_status('Nube anterior restaurada.', 'success')

    def _pick_bg_color(self, event=None):
        """Abre selector de color para el fondo. (H2 - Reconocimiento)"""
        result = colorchooser.askcolor(
            color=self._bg_color.get(),
            title='Seleccionar color de fondo',
            parent=self
        )
        if result[1]:
            self._bg_color.set(result[1])
            self._bg_swatch.configure(bg=result[1])

    def _on_escape(self):
        if self._processing:
            self._set_status('Proceso en curso, por favor espera…', 'info')

    #  Helpers de UI 

    def _start_progress(self, msg: str = ''):
        self._progress.start(12)
        self._progress_lbl.configure(text=msg)

    def _stop_progress(self):
        self._progress.stop()
        self._progress_lbl.configure(text='')

    def _set_status(self, msg: str, kind: str = 'info'):
        """Actualiza la barra de estado con icono y mensaje. (H1, H9)"""
        icons = {'info': '', 'success': '', 'error': '', 'warning': ''}
        self._status_var.set(msg)
        self._status_icon.set(icons.get(kind, ''))

    def _on_close(self):
        plt.close('all')
        self.destroy()


print(' Clase WordCloudApp cargada')

 Clase WordCloudApp cargada


---
## Ejecución

> **Nota:** La ventana de tkinter se abrirá como una ventana de escritorio separada. Asegúrate de tener un entorno gráfico disponible (display X11/Wayland en Linux, o ejecutar desde la terminal en Windows/macOS).
```bash
python main.ipynb
```

In [ ]:
if __name__ == '__main__':
    app = WordCloudApp()
    app.mainloop()

---

## Prueba de los lectores de archivo (sin GUI)

Puedes probar los lectores de archivo individualmente sin abrir la ventana de la aplicación.

In [ ]:
# Demo: generar una nube de palabras de muestra en el notebook
import matplotlib.pyplot as plt

texto_demo = """
Python es un lenguaje de programación interpretado cuya filosofía hace hincapié
en una sintaxis que favorezca un código legible. Se trata de un lenguaje de programación
multiparadigma, ya que soporta orientación a objetos, programación imperativa y, en menor
medida, programación funcional. Es administrado por la Python Software Foundation.
Python fue diseñado para ser leído con facilidad. Sus convenciones de diseño son visibles
en su filosofía explícita en el Zen de Python.
La comunidad de Python es enorme y activa. Hay bibliotecas para todo:
ciencia de datos, inteligencia artificial, desarrollo web, automatización.
Python es usado por empresas como Google, Netflix, Instagram y Spotify.
"""

wc_demo = generate_wordcloud(
    texto_demo,
    max_words=60,
    bg_color='white',
    colormap='viridis',
    lang='Español',
    width=800,
    height=400,
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(wc_demo, interpolation='bilinear')
ax.axis('off')
ax.set_title('Demo — Nube de palabras (texto sobre Python)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

print(f'Palabras en la nube: {len(wc_demo.words_)}')
print('Top 10:')
for w, freq in list(wc_demo.words_.items())[:10]:
    print(f'  {w:20s} {freq:.3f}')